[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/05_ONNX_Operators_and_OpSets/04_Operator_Schemas/Operator_Schemas_Apply.ipynb)

# 5.4 Operator Schemas — Hands-On Practice

## Table of Contents
1. [Exercise 1: Query Operator Schemas Programmatically](#exercise-1)
2. [Exercise 2: List All Inputs/Outputs/Attributes for Common Ops](#exercise-2)
3. [Exercise 3: Type Constraint Exploration](#exercise-3)
4. [Exercise 4: Manual Shape Inference Computation](#exercise-4)
5. [Exercise 5: Shape Inference Verification with ONNX](#exercise-5)
6. [Exercise 6: Schema Comparison Across Opset Versions](#exercise-6)
7. [Exercise 7: Optionality and Variadic Parameter Exploration](#exercise-7)
8. [Challenge: Build a Schema Validator Tool](#challenge)
9. [Visualizations: Schema Statistics Dashboard](#viz)
10. [Summary](#summary)

In [ ]:
# Install dependencies
!pip install onnx onnxruntime numpy matplotlib --quiet

In [ ]:
import onnx
from onnx import defs, helper, TensorProto, checker, shape_inference
from onnx import numpy_helper
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter

opset = defs.onnx_opset_version()
print(f"ONNX version: {onnx.__version__}")
print(f"Default opset version: {opset}")

<a id='exercise-1'></a>
## Exercise 1: Query Operator Schemas Programmatically

In this exercise, you will build a utility function to query and display complete schema information for any ONNX operator. This is the foundation skill for all schema-related work.

The key API is `onnx.defs.get_schema(op_type, opset_version, domain)` which returns an `OpSchema` object. From this object, you can access all components: inputs, outputs, attributes, type constraints, and documentation.

Understanding how to programmatically explore schemas is essential for:
- Debugging export errors ("what inputs does this op expect?")
- Building automated validation tools
- Understanding type compatibility before mixed-precision deployment
- Generating documentation or tooling around ONNX models

In [ ]:
def query_full_schema(op_type, opset_ver=None, domain=""):
    """
    Query and display the complete schema for an ONNX operator.
    Returns the schema object for further inspection.
    """
    if opset_ver is None:
        opset_ver = opset
    
    s = defs.get_schema(op_type, opset_ver, domain)
    
    print(f"╔{'═'*62}╗")
    print(f"║  {s.name:^58s}  ║")
    print(f"╠{'═'*62}╣")
    print(f"║  Domain:        {domain if domain else '(default)':42s}  ║")
    print(f"║  Since Version: {s.since_version:<42d}  ║")
    print(f"║  Queried at:    opset {opset_ver:<37d}  ║")
    print(f"║  Deprecated:    {str(s.deprecated):<42s}  ║")
    print(f"╚{'═'*62}╝")
    
    # Inputs
    print(f"\n  ┌─ INPUTS ({len(s.inputs)})")
    for i, p in enumerate(s.inputs):
        opt = str(p.option)
        marker = "●" if "Single" in opt else ("○" if "Optional" in opt else "◆")
        print(f"  │  {marker} [{i}] {p.name:15s} type_var={p.type_str:8s} ({opt.split('.')[-1]})")
    
    # Outputs
    print(f"  │")
    print(f"  ├─ OUTPUTS ({len(s.outputs)})")
    for i, p in enumerate(s.outputs):
        opt = str(p.option)
        marker = "●" if "Single" in opt else ("○" if "Optional" in opt else "◆")
        print(f"  │  {marker} [{i}] {p.name:15s} type_var={p.type_str:8s} ({opt.split('.')[-1]})")
    
    # Type Constraints
    print(f"  │")
    print(f"  ├─ TYPE CONSTRAINTS ({len(s.type_constraints)})")
    for tc in s.type_constraints:
        types = sorted(tc.allowed_type_strs)
        short_types = [t.replace('tensor(', '').replace(')', '') for t in types]
        if len(short_types) > 6:
            display = ', '.join(short_types[:6]) + f', ... ({len(types)} total)'
        else:
            display = ', '.join(short_types)
        print(f"  │  {tc.type_param_str}: [{display}]")
    
    # Attributes
    type_names = {1:'FLOAT', 2:'INT', 3:'STRING', 4:'TENSOR', 5:'GRAPH',
                  6:'FLOATS', 7:'INTS', 8:'STRINGS', 9:'TENSORS', 10:'GRAPHS'}
    print(f"  │")
    print(f"  └─ ATTRIBUTES ({len(s.attributes)})")
    for name, attr in sorted(s.attributes.items()):
        type_name = type_names.get(int(attr.type), '?')
        req = "REQUIRED" if attr.required else "optional"
        print(f"     • {name:20s} {type_name:8s} [{req}]")
    
    print()
    return s

# Query several operators
_ = query_full_schema("Conv")

In [ ]:
_ = query_full_schema("MatMul")

In [ ]:
_ = query_full_schema("Gather")

<a id='exercise-2'></a>
## Exercise 2: List All Inputs/Outputs/Attributes for Common Ops

Now let's systematically catalog the interface of commonly-used operators. This exercise builds a reference table showing the "shape" of each operator — how many parameters it takes and what kind they are.

This is useful for:
- Quickly debugging "wrong number of inputs" errors
- Understanding which ops are simple (element-wise) vs. complex (many attributes)
- Planning model construction when using `onnx.helper.make_node`

The complexity metric we use here is:

$$\text{complexity}(\text{op}) = |\text{inputs}| + |\text{outputs}| + |\text{attributes}|$$

Simple element-wise ops (Relu, Sigmoid) have complexity around 2-3, while complex ops like LSTM can have complexity > 20.

In [ ]:
# Comprehensive operator catalog
common_ops = [
    # Element-wise
    "Relu", "Sigmoid", "Tanh", "Exp", "Log", "Sqrt",
    # Binary
    "Add", "Sub", "Mul", "Div", "Pow",
    # Reduction
    "ReduceMean", "ReduceSum", "ReduceMax",
    # Linear algebra
    "MatMul", "Gemm",
    # Convolution
    "Conv", "ConvTranspose",
    # Normalization
    "BatchNormalization", "LayerNormalization",
    # Pooling
    "MaxPool", "AveragePool", "GlobalAveragePool",
    # Shape manipulation
    "Reshape", "Transpose", "Squeeze", "Unsqueeze", "Flatten",
    # Data movement
    "Gather", "Scatter", "Concat", "Split", "Slice",
    # Activation
    "Softmax", "LogSoftmax",
    # Recurrent
    "LSTM", "GRU",
]

print(f"{'Operator':<22} {'Inputs':<8} {'Outputs':<9} {'Attrs':<7} {'Since':<7} {'Complexity'}")
print("─" * 70)

results = []
for op in common_ops:
    try:
        s = defs.get_schema(op, opset, "")
        n_in = len(s.inputs)
        n_out = len(s.outputs)
        n_attr = len(s.attributes)
        complexity = n_in + n_out + n_attr
        bar = "█" * complexity
        print(f"{op:<22} {n_in:<8} {n_out:<9} {n_attr:<7} v{s.since_version:<5} {bar} ({complexity})")
        results.append({'op': op, 'inputs': n_in, 'outputs': n_out, 
                       'attrs': n_attr, 'complexity': complexity})
    except Exception as e:
        print(f"{op:<22} NOT FOUND at opset {opset}")

print(f"\n{'─'*70}")
print(f"Total operators cataloged: {len(results)}")
print(f"Most complex: {max(results, key=lambda x: x['complexity'])['op']}")
print(f"Simplest: {min(results, key=lambda x: x['complexity'])['op']}")

<a id='exercise-3'></a>
## Exercise 3: Type Constraint Exploration

Type constraints are the ONNX type system. Understanding them is critical for:

- **Mixed precision**: Can I use float16 inputs with this operator?
- **Quantization**: Which ops support int8/uint8?
- **Index operations**: Why does Gather need int64 indices?

In this exercise, we'll systematically explore which types each operator supports and identify patterns in type constraint design.

Key questions to answer:
- Which operators support the widest range of types? (most polymorphic)
- Which operators are restricted to floating-point only?
- How do operators handle mixed-type scenarios (data + indices)?

In [ ]:
# Analyze type support patterns
type_categories = {
    'float16': 'float', 'float': 'float', 'double': 'float', 'bfloat16': 'float',
    'int8': 'int', 'int16': 'int', 'int32': 'int', 'int64': 'int',
    'uint8': 'uint', 'uint16': 'uint', 'uint32': 'uint', 'uint64': 'uint',
    'bool': 'bool', 'string': 'string', 'complex64': 'complex', 'complex128': 'complex',
}

def categorize_types(allowed_type_strs):
    """Categorize allowed types into groups."""
    categories = set()
    for t in allowed_type_strs:
        clean = t.replace('tensor(', '').replace(')', '')
        cat = type_categories.get(clean, 'other')
        categories.add(cat)
    return categories

ops_to_analyze = ["Add", "MatMul", "Conv", "Relu", "Sigmoid", "Softmax",
                  "Gather", "Where", "Cast", "Reshape", "Transpose",
                  "Equal", "Less", "ReduceMean", "Concat"]

print(f"{'Operator':<16} {'#Types':<8} {'Float':<7} {'Int':<5} {'UInt':<6} {'Bool':<6} {'Str':<5} {'Type Vars'}")
print("─" * 80)

for op in ops_to_analyze:
    try:
        s = defs.get_schema(op, opset, "")
        all_types = set()
        type_vars = []
        for tc in s.type_constraints:
            all_types.update(tc.allowed_type_strs)
            type_vars.append(tc.type_param_str)
        
        cats = categorize_types(all_types)
        has_float = '✓' if 'float' in cats else '✗'
        has_int = '✓' if 'int' in cats else '✗'
        has_uint = '✓' if 'uint' in cats else '✗'
        has_bool = '✓' if 'bool' in cats else '✗'
        has_str = '✓' if 'string' in cats else '✗'
        
        print(f"{op:<16} {len(all_types):<8} {has_float:<7} {has_int:<5} {has_uint:<6} {has_bool:<6} {has_str:<5} {type_vars}")
    except Exception:
        print(f"{op:<16} NOT FOUND")

In [ ]:
# Deep dive: Compare type constraints between operations that share type vars differently
print("\n" + "═" * 60)
print("  MULTI-TYPE-VARIABLE OPERATORS")
print("═" * 60)
print("\n  These operators use multiple type variables, allowing")
print("  different input types (e.g., data vs indices).\n")

multi_type_ops = []
for op in common_ops:
    try:
        s = defs.get_schema(op, opset, "")
        if len(s.type_constraints) >= 2:
            multi_type_ops.append(op)
    except Exception:
        pass

for op in multi_type_ops[:8]:
    s = defs.get_schema(op, opset, "")
    print(f"  {op}:")
    for tc in s.type_constraints:
        short_types = [t.replace('tensor(', '').replace(')', '') for t in sorted(tc.allowed_type_strs)[:5]]
        more = f" +{len(tc.allowed_type_strs)-5} more" if len(tc.allowed_type_strs) > 5 else ""
        # Which params use this type var?
        params_using = [p.name for p in list(s.inputs) + list(s.outputs) if p.type_str == tc.type_param_str]
        print(f"    {tc.type_param_str:6s} → [{', '.join(short_types)}{more}]")
        print(f"           used by: {params_using}")
    print()

<a id='exercise-4'></a>
## Exercise 4: Manual Shape Inference Computation

In this exercise, you will manually compute output shapes for several operators using their formal shape inference rules, then verify your answers.

### Shape Inference Formulas

**Conv2D:**
$$H_{out} = \left\lfloor \frac{H_{in} + p_t + p_b - d_h(k_h - 1) - 1}{s_h} \right\rfloor + 1$$

**MaxPool/AveragePool:** Same formula as Conv (operates on spatial dims only).

**MatMul:** Batch broadcasting + $[\ldots, M, K] \times [\ldots, K, N] \rightarrow [\ldots, M, N]$

**Reshape:** Element conservation with $-1$ inference: $d_{-1} = \frac{\prod d_i^{\text{in}}}{\prod_{j \neq -1} d_j^{\text{out}}}$

**Transpose:** Permutation of dimensions: $\text{shape}_{out}[i] = \text{shape}_{in}[\text{perm}[i]]$

**Concat:** All dims same except along axis: $d_{\text{axis}}^{out} = \sum_i d_{\text{axis}}^{(i)}$

In [ ]:
# Manual shape inference implementations

def manual_conv_shape(input_shape, weight_shape, strides, pads, dilations, group=1):
    """ONNX Conv shape inference."""
    N, C_in, H_in, W_in = input_shape
    C_out = weight_shape[0]
    kH, kW = weight_shape[2], weight_shape[3]
    sH, sW = strides
    dH, dW = dilations
    pt, pl, pb, pr = pads  # ONNX order: [top, left, bottom, right]
    
    H_out = (H_in + pt + pb - dH * (kH - 1) - 1) // sH + 1
    W_out = (W_in + pl + pr - dW * (kW - 1) - 1) // sW + 1
    return [N, C_out, H_out, W_out]

def manual_pool_shape(input_shape, kernel_shape, strides, pads):
    """ONNX MaxPool/AveragePool shape inference."""
    N, C, H_in, W_in = input_shape
    kH, kW = kernel_shape
    sH, sW = strides
    pt, pl, pb, pr = pads
    
    H_out = (H_in + pt + pb - kH) // sH + 1
    W_out = (W_in + pl + pr - kW) // sW + 1
    return [N, C, H_out, W_out]

def manual_transpose_shape(input_shape, perm):
    """ONNX Transpose shape inference."""
    return [input_shape[p] for p in perm]

def manual_concat_shape(input_shapes, axis):
    """ONNX Concat shape inference."""
    # Negative axis
    ndim = len(input_shapes[0])
    if axis < 0:
        axis += ndim
    
    result = list(input_shapes[0])
    result[axis] = sum(s[axis] for s in input_shapes)
    return result

# Test cases — compute manually then verify
print("Manual Shape Inference Exercises")
print("═" * 60)

# Case 1: VGG-style Conv
result = manual_conv_shape([1, 64, 224, 224], [128, 64, 3, 3], [1, 1], [1, 1, 1, 1], [1, 1])
print(f"\n  Case 1 — VGG Conv (3×3, stride 1, pad 1):")
print(f"    Input:  [1, 64, 224, 224]")
print(f"    Weight: [128, 64, 3, 3]")
print(f"    Output: {result}")

# Case 2: ResNet downsample
result = manual_conv_shape([1, 64, 56, 56], [128, 64, 3, 3], [2, 2], [1, 1, 1, 1], [1, 1])
print(f"\n  Case 2 — ResNet downsample (3×3, stride 2, pad 1):")
print(f"    Input:  [1, 64, 56, 56]")
print(f"    Weight: [128, 64, 3, 3]")
print(f"    Output: {result}")

# Case 3: Dilated convolution
result = manual_conv_shape([1, 64, 32, 32], [64, 64, 3, 3], [1, 1], [2, 2, 2, 2], [2, 2])
print(f"\n  Case 3 — Dilated conv (3×3, dilation 2, pad 2):")
print(f"    Input:  [1, 64, 32, 32]")
print(f"    Weight: [64, 64, 3, 3]")
print(f"    Output: {result}")
print(f"    Note: dilated 3×3 has effective receptive field of 5×5")

# Case 4: Pooling
result = manual_pool_shape([1, 64, 112, 112], [3, 3], [2, 2], [1, 1, 1, 1])
print(f"\n  Case 4 — MaxPool (3×3, stride 2, pad 1):")
print(f"    Input:  [1, 64, 112, 112]")
print(f"    Output: {result}")

# Case 5: Transpose (attention reshape)
result = manual_transpose_shape([8, 12, 64, 64], [0, 2, 1, 3])
print(f"\n  Case 5 — Transpose perm=[0,2,1,3] (swap heads and seq):")
print(f"    Input:  [8, 12, 64, 64]")
print(f"    Output: {result}")

# Case 6: Concat
result = manual_concat_shape([[2, 3, 4], [2, 5, 4], [2, 1, 4]], axis=1)
print(f"\n  Case 6 — Concat axis=1:")
print(f"    Inputs: [2,3,4], [2,5,4], [2,1,4]")
print(f"    Output: {result}")

<a id='exercise-5'></a>
## Exercise 5: Shape Inference Verification with ONNX

Now let's verify our manual computations against ONNX's built-in shape inference engine. This exercise builds models, runs `onnx.shape_inference.infer_shapes()`, and compares the results with our manual calculations.

This verification pattern is extremely useful in practice:
1. Compute expected shape manually (or from architecture docs)
2. Build the ONNX subgraph
3. Run shape inference
4. Assert shapes match

If they don't match, you've found a bug in either your understanding or your export code.

In [ ]:
def verify_shape(description, expected_shape, node, inputs, initializers=None):
    """
    Verify that ONNX shape inference matches expected output shape.
    """
    if initializers is None:
        initializers = []
    
    output = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    graph = helper.make_graph([node], "verify", inputs, [output], initializer=initializers)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    
    inferred = shape_inference.infer_shapes(model)
    out_shape = [d.dim_value for d in inferred.graph.output[0].type.tensor_type.shape.dim]
    
    match = out_shape == expected_shape
    status = "✓ MATCH" if match else "✗ MISMATCH"
    print(f"  {status}  {description}")
    print(f"           Expected: {expected_shape}")
    print(f"           Got:      {out_shape}")
    if not match:
        print(f"           ERROR: shapes differ!")
    print()
    return match

print("Shape Inference Verification")
print("═" * 60)

# Verify 1: Conv
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 64, 56, 56])
W = numpy_helper.from_array(np.zeros((128, 64, 3, 3), dtype=np.float32), name="W")
conv_node = helper.make_node("Conv", ["X", "W"], ["Y"],
                             kernel_shape=[3, 3], strides=[2, 2], pads=[1, 1, 1, 1])
verify_shape("Conv 3×3 stride 2 pad 1", [1, 128, 28, 28], conv_node, [X], [W])

# Verify 2: MatMul
A = helper.make_tensor_value_info("X", TensorProto.FLOAT, [8, 64, 512])
B_init = numpy_helper.from_array(np.zeros((512, 256), dtype=np.float32), name="B_mat")
matmul_node = helper.make_node("MatMul", ["X", "B_mat"], ["Y"])
verify_shape("MatMul [8,64,512]×[512,256]", [8, 64, 256], matmul_node, [A], [B_init])

# Verify 3: Transpose
T_in = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 12, 64, 32])
trans_node = helper.make_node("Transpose", ["X"], ["Y"], perm=[0, 2, 1, 3])
verify_shape("Transpose perm=[0,2,1,3]", [2, 64, 12, 32], trans_node, [T_in])

# Verify 4: Reshape
R_in = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 12, 64])
shape_val = numpy_helper.from_array(np.array([4, -1], dtype=np.int64), name="shape")
reshape_node = helper.make_node("Reshape", ["X", "shape"], ["Y"])
verify_shape("Reshape [4,12,64]→[4,-1]", [4, 768], reshape_node, [R_in], [shape_val])

# Verify 5: AveragePool
P_in = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 256, 14, 14])
pool_node = helper.make_node("AveragePool", ["X"], ["Y"],
                             kernel_shape=[2, 2], strides=[2, 2])
verify_shape("AvgPool 2×2 stride 2", [1, 256, 7, 7], pool_node, [P_in])

<a id='exercise-6'></a>
## Exercise 6: Schema Comparison Across Opset Versions

Operators evolve across opset versions — attributes are added, removed, or have their semantics changed. Understanding these changes is crucial when:

- Upgrading a model from one opset to another
- Debugging "attribute not found" errors after export
- Understanding why the same operator behaves differently across frameworks

In this exercise, we'll track how specific operators changed their schemas over time.

Notable historical changes:
- **Softmax (opset 13)**: Changed axis default from 1 to -1
- **BatchNormalization (opset 15)**: Changed training mode outputs  
- **Squeeze/Unsqueeze (opset 13)**: Moved `axes` from attribute to input
- **ReduceMean (opset 18)**: Moved `axes` from attribute to input

In [ ]:
def compare_schema_versions(op_type, versions, domain=""):
    """
    Compare an operator's schema across multiple opset versions.
    Shows what changed between versions.
    """
    print(f"\n{'═'*70}")
    print(f"  Schema Evolution: {op_type}")
    print(f"{'═'*70}")
    
    prev_attrs = None
    prev_inputs = None
    prev_outputs = None
    prev_version = None
    
    for v in versions:
        try:
            s = defs.get_schema(op_type, v, domain)
            curr_attrs = set(s.attributes.keys())
            curr_inputs = [(p.name, str(p.option)) for p in s.inputs]
            curr_outputs = [(p.name, str(p.option)) for p in s.outputs]
            
            print(f"\n  Opset {v:2d} (schema since v{s.since_version}):")
            print(f"    Inputs:  {[p.name for p in s.inputs]}")
            print(f"    Outputs: {[p.name for p in s.outputs]}")
            print(f"    Attrs:   {sorted(curr_attrs)}")
            
            # Show differences from previous version
            if prev_attrs is not None:
                added = curr_attrs - prev_attrs
                removed = prev_attrs - curr_attrs
                input_names_curr = [n for n, _ in curr_inputs]
                input_names_prev = [n for n, _ in prev_inputs]
                
                changes = []
                if added:
                    changes.append(f"attrs added: {added}")
                if removed:
                    changes.append(f"attrs removed: {removed}")
                if input_names_curr != input_names_prev:
                    changes.append(f"inputs changed: {input_names_prev} → {input_names_curr}")
                if len(curr_outputs) != len(prev_outputs):
                    changes.append(f"output count: {len(prev_outputs)} → {len(curr_outputs)}")
                
                if changes:
                    print(f"    ⚠ Changes from opset {prev_version}:")
                    for c in changes:
                        print(f"      → {c}")
                else:
                    print(f"    (no schema change from opset {prev_version})")
            
            prev_attrs = curr_attrs
            prev_inputs = curr_inputs
            prev_outputs = curr_outputs
            prev_version = v
            
        except Exception as e:
            print(f"\n  Opset {v:2d}: NOT AVAILABLE ({e})")
    print()

# Track evolution of key operators
compare_schema_versions("Softmax", [1, 11, 13, 17])
compare_schema_versions("BatchNormalization", [1, 9, 14, 17])
compare_schema_versions("Squeeze", [1, 11, 13, 17])

In [ ]:
# Find all opset version transitions for key operators
def find_schema_revisions(op_type, domain=""):
    """Find all opset versions where this operator's schema changed."""
    revisions = []
    all_schemas = defs.get_all_schemas_with_history()
    for s in all_schemas:
        if s.name == op_type and s.domain == domain:
            revisions.append(s.since_version)
    return sorted(revisions)

key_ops = ["Add", "Conv", "Relu", "Softmax", "BatchNormalization", 
           "Reshape", "Squeeze", "Unsqueeze", "MatMul", "Gemm",
           "ReduceMean", "Gather", "Pad", "Resize", "Slice"]

print(f"{'Operator':<25} {'Revisions':<40} {'#Revs'}")
print("─" * 75)
for op in key_ops:
    revs = find_schema_revisions(op)
    print(f"{op:<25} {str(revs):<40} {len(revs)}")

print(f"\nMost revised operators have the most complex version compatibility stories.")
print(f"Each revision = potential behavior change when upgrading opsets.")

<a id='exercise-7'></a>
## Exercise 7: Optionality and Variadic Parameter Exploration

Understanding parameter optionality is essential for correctly constructing ONNX nodes. In this exercise, we'll:

1. Identify all operators with optional inputs
2. Identify all operators with variadic inputs/outputs
3. Practice building nodes that use optional and variadic parameters correctly

Common pitfalls:
- Forgetting that Conv's bias is optional (not all exports include it)
- Not providing enough inputs for variadic parameters
- Using empty string `""` to skip optional inputs at incorrect positions

In [ ]:
# Survey optionality across all standard operators
optional_input_ops = []
variadic_input_ops = []
variadic_output_ops = []

all_schemas = defs.get_all_schemas_with_history()
latest = {}
for s in all_schemas:
    if s.domain == "":
        if s.name not in latest or s.since_version > latest[s.name].since_version:
            latest[s.name] = s

for name, s in latest.items():
    has_optional_in = any('Optional' in str(p.option) for p in s.inputs)
    has_variadic_in = any('Variadic' in str(p.option) for p in s.inputs)
    has_variadic_out = any('Variadic' in str(p.option) for p in s.outputs)
    
    if has_optional_in:
        opt_names = [p.name for p in s.inputs if 'Optional' in str(p.option)]
        optional_input_ops.append((name, opt_names))
    if has_variadic_in:
        var_names = [p.name for p in s.inputs if 'Variadic' in str(p.option)]
        variadic_input_ops.append((name, var_names))
    if has_variadic_out:
        var_names = [p.name for p in s.outputs if 'Variadic' in str(p.option)]
        variadic_output_ops.append((name, var_names))

print(f"Operators with OPTIONAL inputs: {len(optional_input_ops)}")
print("─" * 50)
for name, params in sorted(optional_input_ops)[:15]:
    print(f"  {name:25s} optional: {params}")
if len(optional_input_ops) > 15:
    print(f"  ... and {len(optional_input_ops)-15} more")

print(f"\nOperators with VARIADIC inputs: {len(variadic_input_ops)}")
print("─" * 50)
for name, params in sorted(variadic_input_ops)[:10]:
    print(f"  {name:25s} variadic: {params}")

print(f"\nOperators with VARIADIC outputs: {len(variadic_output_ops)}")
print("─" * 50)
for name, params in sorted(variadic_output_ops)[:10]:
    print(f"  {name:25s} variadic: {params}")

In [ ]:
# Practical: Build nodes with variadic inputs (Concat)
print("Variadic Input Example: Concat")
print("═" * 50)

# Concat 4 tensors along axis 1
inputs_vi = [helper.make_tensor_value_info(f"X{i}", TensorProto.FLOAT, [2, s, 4]) 
             for i, s in enumerate([3, 5, 2, 7])]
output_vi = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

concat_node = helper.make_node("Concat", [f"X{i}" for i in range(4)], ["Y"], axis=1)
graph = helper.make_graph([concat_node], "concat_test", inputs_vi, [output_vi])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

inferred = shape_inference.infer_shapes(model)
out_shape = [d.dim_value for d in inferred.graph.output[0].type.tensor_type.shape.dim]
print(f"  Inputs: [2,3,4], [2,5,4], [2,2,4], [2,7,4]")
print(f"  Concat axis=1")
print(f"  Output: {out_shape}")
print(f"  Expected: [2, {3+5+2+7}, 4] = [2, 17, 4]")
checker.check_model(model)
print(f"  ✓ Model valid")

<a id='challenge'></a>
## Challenge: Build a Schema Validator Tool

In this challenge, you'll build a custom schema validator that checks an ONNX node against its schema **before** constructing a full model. This is useful for catching errors early during graph construction.

Your validator should check:
1. ✓ Operator exists at the specified opset
2. ✓ Number of inputs is within valid range (considering Optional/Variadic)
3. ✓ Number of outputs is within valid range
4. ✓ All required attributes are present
5. ✓ Attribute types match the schema

This is essentially reimplementing part of `onnx.checker` but with more detailed error messages.

In [ ]:
class SchemaValidator:
    """Custom ONNX schema validator with detailed diagnostics."""
    
    def __init__(self, opset_version=None, domain=""):
        self.opset = opset_version or defs.onnx_opset_version()
        self.domain = domain
        self.errors = []
        self.warnings = []
    
    def validate_node(self, op_type, input_names, output_names, attributes=None):
        """
        Validate a node specification against its schema.
        Returns (is_valid, errors, warnings).
        """
        self.errors = []
        self.warnings = []
        attributes = attributes or {}
        
        # Check 1: Operator exists
        try:
            schema = defs.get_schema(op_type, self.opset, self.domain)
        except Exception as e:
            self.errors.append(f"Operator '{op_type}' not found at opset {self.opset}: {e}")
            return False, self.errors, self.warnings
        
        # Check 2: Input count
        min_inputs = sum(1 for p in schema.inputs if 'Single' in str(p.option))
        has_variadic_in = any('Variadic' in str(p.option) for p in schema.inputs)
        max_inputs = float('inf') if has_variadic_in else len(schema.inputs)
        
        n_inputs = len(input_names)
        if n_inputs < min_inputs:
            self.errors.append(
                f"Too few inputs: got {n_inputs}, need at least {min_inputs}. "
                f"Required: {[p.name for p in schema.inputs if 'Single' in str(p.option)]}")
        elif n_inputs > max_inputs:
            self.errors.append(
                f"Too many inputs: got {n_inputs}, max is {int(max_inputs)}")
        
        # Check 3: Output count
        min_outputs = sum(1 for p in schema.outputs if 'Single' in str(p.option))
        has_variadic_out = any('Variadic' in str(p.option) for p in schema.outputs)
        max_outputs = float('inf') if has_variadic_out else len(schema.outputs)
        
        n_outputs = len(output_names)
        if n_outputs < min_outputs:
            self.errors.append(
                f"Too few outputs: got {n_outputs}, need at least {min_outputs}")
        elif n_outputs > max_outputs:
            self.errors.append(
                f"Too many outputs: got {n_outputs}, max is {int(max_outputs)}")
        
        # Check 4: Required attributes
        for attr_name, attr_def in schema.attributes.items():
            if attr_def.required and attr_name not in attributes:
                self.errors.append(
                    f"Missing required attribute: '{attr_name}'")
        
        # Check 5: Unknown attributes
        for attr_name in attributes:
            if attr_name not in schema.attributes:
                self.warnings.append(
                    f"Unknown attribute: '{attr_name}' (not in schema)")
        
        is_valid = len(self.errors) == 0
        return is_valid, self.errors, self.warnings
    
    def report(self, op_type, input_names, output_names, attributes=None):
        """Validate and print a formatted report."""
        valid, errors, warnings = self.validate_node(
            op_type, input_names, output_names, attributes)
        
        status = "✓ VALID" if valid else "✗ INVALID"
        print(f"\n  {status}: {op_type}(inputs={input_names}, outputs={output_names})")
        if attributes:
            print(f"           attrs={attributes}")
        for e in errors:
            print(f"    ERROR:   {e}")
        for w in warnings:
            print(f"    WARNING: {w}")
        return valid

# Test the validator
validator = SchemaValidator(opset_version=17)

print("Schema Validator Test Suite")
print("═" * 60)

# Valid cases
validator.report("Relu", ["X"], ["Y"])
validator.report("Conv", ["X", "W"], ["Y"], {"kernel_shape": [3, 3]})
validator.report("Conv", ["X", "W", "B"], ["Y"], {"kernel_shape": [3, 3]})
validator.report("Concat", ["X1", "X2", "X3"], ["Y"], {"axis": 1})

# Invalid cases
validator.report("Relu", ["X", "extra"], ["Y"])  # too many inputs
validator.report("Relu", [], ["Y"])  # too few inputs
validator.report("Concat", ["X1", "X2"], ["Y"], {})  # missing required 'axis'
validator.report("Conv", ["X", "W"], ["Y"], {"kernel_shape": [3,3], "magic": 42})  # unknown attr

In [ ]:
# Extended challenge: Validate an entire model graph node-by-node
def validate_model_graph(model, verbose=True):
    """
    Validate every node in a model against its schema.
    Returns list of (node_idx, node_name, errors) for invalid nodes.
    """
    opset_ver = model.opset_import[0].version
    validator = SchemaValidator(opset_version=opset_ver)
    
    issues = []
    for i, node in enumerate(model.graph.node):
        attrs = {a.name: True for a in node.attribute}
        valid, errors, warnings = validator.validate_node(
            node.op_type, list(node.input), list(node.output), attrs)
        
        if not valid or warnings:
            issues.append((i, node.op_type, errors, warnings))
            if verbose:
                status = "✗" if not valid else "⚠"
                print(f"  {status} Node {i} ({node.op_type}): "
                      f"{'; '.join(errors + warnings)}")
    
    if not issues:
        print(f"  ✓ All {len(model.graph.node)} nodes pass schema validation")
    
    return issues

# Build a valid model and validate it
print("\nFull Model Validation:")
print("─" * 50)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 3, 32, 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 10])

W1 = numpy_helper.from_array(np.random.randn(16, 3, 3, 3).astype(np.float32), name="W1")
W2 = numpy_helper.from_array(np.random.randn(10, 16*16*16).astype(np.float32), name="W2")
B2 = numpy_helper.from_array(np.random.randn(10).astype(np.float32), name="B2")
shape_flat = numpy_helper.from_array(np.array([1, -1], dtype=np.int64), name="flat_shape")

nodes = [
    helper.make_node("Conv", ["X", "W1"], ["conv_out"], kernel_shape=[3, 3], strides=[2, 2], pads=[1,1,1,1]),
    helper.make_node("Relu", ["conv_out"], ["relu_out"]),
    helper.make_node("Reshape", ["relu_out", "flat_shape"], ["flat_out"]),
    helper.make_node("Gemm", ["flat_out", "W2", "B2"], ["Y"], transB=1),
]

graph = helper.make_graph(nodes, "demo", [X], [Y], initializer=[W1, W2, B2, shape_flat])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

validate_model_graph(model)

<a id='viz'></a>
## Visualizations: Schema Statistics Dashboard

Let's create comprehensive visualizations that reveal patterns in the ONNX operator schema ecosystem. These charts help build intuition about operator complexity and design patterns.

In [ ]:
# Collect comprehensive schema stats
all_schemas_list = defs.get_all_schemas_with_history()
latest_schemas = {}
for s in all_schemas_list:
    if s.domain == "":
        if s.name not in latest_schemas or s.since_version > latest_schemas[s.name].since_version:
            latest_schemas[s.name] = s

stats = []
for name, s in latest_schemas.items():
    n_type_vars = len(s.type_constraints)
    total_types = sum(len(tc.allowed_type_strs) for tc in s.type_constraints)
    
    # Count optional/variadic
    n_optional_in = sum(1 for p in s.inputs if 'Optional' in str(p.option))
    n_variadic_in = sum(1 for p in s.inputs if 'Variadic' in str(p.option))
    n_required_attrs = sum(1 for a in s.attributes.values() if a.required)
    n_optional_attrs = len(s.attributes) - n_required_attrs
    
    stats.append({
        'name': name,
        'since': s.since_version,
        'n_inputs': len(s.inputs),
        'n_outputs': len(s.outputs),
        'n_attrs': len(s.attributes),
        'n_required_attrs': n_required_attrs,
        'n_optional_attrs': n_optional_attrs,
        'n_type_vars': n_type_vars,
        'total_types': total_types,
        'n_optional_in': n_optional_in,
        'n_variadic_in': n_variadic_in,
    })

print(f"Collected stats for {len(stats)} operators")

In [ ]:
# Dashboard visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Operators introduced per opset version
ax = axes[0, 0]
since_counts = Counter(s['since'] for s in stats)
versions_sorted = sorted(since_counts.keys())
counts = [since_counts[v] for v in versions_sorted]
ax.bar(versions_sorted, counts, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Opset Version')
ax.set_ylabel('Operators Introduced')
ax.set_title('New Operators per Opset', fontweight='bold')
# Cumulative line
ax2 = ax.twinx()
cumulative = np.cumsum(counts)
ax2.plot(versions_sorted, cumulative, 'r-o', markersize=3, linewidth=1.5)
ax2.set_ylabel('Cumulative Total', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# 2. Attribute count distribution
ax = axes[0, 1]
attr_vals = [s['n_attrs'] for s in stats]
ax.hist(attr_vals, bins=range(0, max(attr_vals)+2), color='darkorange', 
        alpha=0.8, edgecolor='white')
ax.axvline(np.mean(attr_vals), color='red', linestyle='--', 
           label=f'μ={np.mean(attr_vals):.1f}')
ax.set_xlabel('Number of Attributes')
ax.set_ylabel('Count')
ax.set_title('Attribute Count Distribution', fontweight='bold')
ax.legend()

# 3. Required vs Optional attributes
ax = axes[0, 2]
req_attrs = [s['n_required_attrs'] for s in stats if s['n_attrs'] > 0]
opt_attrs = [s['n_optional_attrs'] for s in stats if s['n_attrs'] > 0]
ax.scatter(req_attrs, opt_attrs, alpha=0.5, c='purple', s=30)
ax.set_xlabel('Required Attributes')
ax.set_ylabel('Optional Attributes')
ax.set_title('Required vs Optional Attrs', fontweight='bold')
ax.plot([0, max(req_attrs)], [0, max(req_attrs)], 'k--', alpha=0.3, label='equal line')
ax.legend()

# 4. Type polymorphism distribution
ax = axes[1, 0]
type_vals = [s['total_types'] for s in stats if s['total_types'] > 0]
ax.hist(type_vals, bins=20, color='seagreen', alpha=0.8, edgecolor='white')
ax.set_xlabel('Total Allowed Types (sum across constraints)')
ax.set_ylabel('Count')
ax.set_title('Type Polymorphism Distribution', fontweight='bold')

# 5. Input/Output count distribution
ax = axes[1, 1]
in_vals = [s['n_inputs'] for s in stats]
out_vals = [s['n_outputs'] for s in stats]
width = 0.35
x = np.arange(max(max(in_vals), max(out_vals)) + 1)
in_hist = [in_vals.count(i) for i in x]
out_hist = [out_vals.count(i) for i in x]
ax.bar(x - width/2, in_hist, width, label='Inputs', color='steelblue', alpha=0.8)
ax.bar(x + width/2, out_hist, width, label='Outputs', color='coral', alpha=0.8)
ax.set_xlabel('Count')
ax.set_ylabel('Number of Operators')
ax.set_title('Input vs Output Count', fontweight='bold')
ax.legend()
ax.set_xlim(-0.5, 10.5)

# 6. Optionality breakdown (pie chart)
ax = axes[1, 2]
n_all_single = sum(1 for s in stats if s['n_optional_in'] == 0 and s['n_variadic_in'] == 0)
n_has_optional = sum(1 for s in stats if s['n_optional_in'] > 0)
n_has_variadic = sum(1 for s in stats if s['n_variadic_in'] > 0)
labels = ['All Single', 'Has Optional', 'Has Variadic']
sizes = [n_all_single, n_has_optional, n_has_variadic]
colors_pie = ['#4a90d9', '#f5a623', '#d0021b']
ax.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.0f%%', 
       startangle=90, textprops={'fontsize': 9})
ax.set_title('Input Optionality Modes', fontweight='bold')

plt.tight_layout()
plt.suptitle('ONNX Operator Schema Statistics Dashboard', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Type support heatmap for key operators
key_ops_for_heatmap = [
    "Add", "Sub", "Mul", "Div", "MatMul", "Conv", "Gemm",
    "Relu", "Sigmoid", "Softmax", "Gather", "Reshape",
    "Concat", "Slice", "Where", "Equal", "Cast"
]

type_categories_display = [
    'float16', 'bfloat16', 'float', 'double',
    'int8', 'int16', 'int32', 'int64',
    'uint8', 'uint16', 'uint32', 'uint64',
    'bool', 'string'
]

# Build support matrix
support_matrix = np.zeros((len(key_ops_for_heatmap), len(type_categories_display)))

for i, op in enumerate(key_ops_for_heatmap):
    try:
        s = defs.get_schema(op, opset, "")
        all_allowed = set()
        for tc in s.type_constraints:
            all_allowed.update(tc.allowed_type_strs)
        
        for j, dtype in enumerate(type_categories_display):
            if f'tensor({dtype})' in all_allowed:
                support_matrix[i, j] = 1
    except Exception:
        pass

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(support_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(type_categories_display)))
ax.set_xticklabels(type_categories_display, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(key_ops_for_heatmap)))
ax.set_yticklabels(key_ops_for_heatmap, fontsize=10)

# Add cell annotations
for i in range(len(key_ops_for_heatmap)):
    for j in range(len(type_categories_display)):
        text = '✓' if support_matrix[i, j] == 1 else ''
        ax.text(j, i, text, ha='center', va='center', fontsize=8)

ax.set_title('Element Type Support Matrix\n(across all type constraints per operator)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Element Type')
ax.set_ylabel('Operator')

# Add type counts on the right
for i in range(len(key_ops_for_heatmap)):
    count = int(support_matrix[i].sum())
    ax.text(len(type_categories_display) + 0.3, i, f'{count}/{len(type_categories_display)}', 
            va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  • Arithmetic ops (Add, Mul) support the widest type range")
print("  • Activation ops (Relu, Sigmoid, Softmax) are float-only")
print("  • Shape ops (Reshape, Concat) are type-agnostic (all types)")
print("  • Comparison ops (Equal) output bool regardless of input type")

<a id='summary'></a>
## Summary

### What You Practiced

| Exercise | Skill | Key Takeaway |
|----------|-------|-------------|
| 1. Schema Query | `defs.get_schema()` API | Foundation for all schema work |
| 2. I/O/Attr Catalog | Operator interface survey | Complexity varies 2-20+ parameters |
| 3. Type Constraints | Polymorphism analysis | Type vars enforce consistency |
| 4. Manual Shape Inference | Formula application | Conv, MatMul, Reshape rules |
| 5. Shape Verification | `shape_inference.infer_shapes()` | Automated verification |
| 6. Version Comparison | Schema evolution tracking | Ops change across opsets |
| 7. Optionality | Optional/Variadic handling | Correct node construction |
| Challenge | Custom validator | Schema-aware tooling |

### Key Formulas

- **Conv**: $H_{out} = \lfloor (H_{in} + p_t + p_b - d(k-1) - 1) / s \rfloor + 1$
- **MatMul**: $[\ldots, M, K] \times [\ldots, K, N] \rightarrow [\text{broadcast}(\ldots), M, N]$
- **Reshape**: $\prod d_i^{in} = \prod d_j^{out}$, with $d_{-1} = \frac{\text{total}}{\prod \text{known}}$

### Next Steps

- Apply schema knowledge to debug real model export failures
- Extend the validator to check type constraints at graph level
- Build tooling that auto-generates test cases from schema definitions
- Explore custom operator schema registration for domain-specific ops